In [1]:
import pandas as pd

df = pd.read_csv('./data/logistics_dirty_dataset.csv')


In [2]:
""" check no. of rows and columns"""
rows, columns = df.shape
print(f'The data contains {rows:,} rows and {columns} columns')

The data contains 150,500 rows and 20 columns


In [3]:
""" Lets check the data types of each columns to see what to adjust"""
df.dtypes

Shipment_ID         object
Order_ID            object
Origin_City         object
Warehouse           object
Destination         object
Weight_kg          float64
Cost_NGN           float64
Distance_km        float64
Delivery_Days        int64
Shipment_Date       object
Delivery_Date       object
Status              object
Vehicle_Type        object
Payment_Method      object
Priority            object
Insurance           object
Driver_Name         object
Remarks             object
Customer_Rating      int64
Notes               object
dtype: object

In [4]:
""" check for null values"""

""" 
for Shipment_ID, the nulls are basically missing values,
 
"""
df.isnull().sum()

Shipment_ID         1476
Order_ID               0
Origin_City            0
Warehouse              0
Destination            0
Weight_kg              0
Cost_NGN               0
Distance_km            0
Delivery_Days          0
Shipment_Date          0
Delivery_Date       3072
Status                 0
Vehicle_Type           0
Payment_Method         0
Priority               0
Insurance              0
Driver_Name            0
Remarks            37337
Customer_Rating        0
Notes                  0
dtype: int64

- Shipment_ID (1,476 nulls): Missing identifiers — 
  rows cannot be uniquely tracked. Will need a decision in cleaning.
- Shipment_Date (2,969 nulls): Invalid dates coerced to NaT — 
  these are the impossible dates like 32-13-2023.
- Delivery_Date (3,072 nulls): Mix of genuinely missing 
  delivery dates and coerced invalid formats.
- Remarks (37,337 nulls): 25% of rows have no remark — 
  combined with only 3 unique values, this column has minimal analytical value.

In [5]:
df.duplicated().sum()

np.int64(500)

500 fully duplicate rows found. These will be dropped in cleaning.

In [12]:
# df['Shipment_Date'].isnull().sum()
# df['Shipment_Date'].unique()
""" the shipment_date & delivery_date columns are objects, so when i tried to convert it into a datetime column, i discovered that some dates are wrongly formartted and some aren't even suppose to be a date which is why i removed them with the errors='coerce' upon converting"""
df['Shipment_Date'] = pd.to_datetime(df['Shipment_Date'], errors='coerce')
df['Delivery_Date'] = pd.to_datetime(df['Delivery_Date'], errors='coerce')


df['Shipment_Date'].min().date()
df['Shipment_Date'].max().date()
print(f'Date range of Shipment is from {df['Shipment_Date'].min().date()} to {df['Shipment_Date'].max().date()}')
df['Delivery_Date'].min().date()
df['Delivery_Date'].max().date()
print(f'Date range of Delivery is from {df['Delivery_Date'].min().date()} to {df['Delivery_Date'].max().date()}')

print(f'Invalid Shipment_Dates (coerced to NaT): {df["Shipment_Date"].isnull().sum()}')
print(f'Invalid Delivery_Dates (coerced to NaT): {df["Delivery_Date"].isnull().sum()}')


Date range of Shipment is from 2022-01-01 to 2025-12-31
Date range of Delivery is from 2022-01-01 to 2026-01-10
Invalid Shipment_Dates (coerced to NaT): 2969
Invalid Delivery_Dates (coerced to NaT): 3072


In [7]:
""" Checking the categorical columns to see if the elements of each column is as it should
I noticed the non-uniformity of similar elements: upper and lower case descripancies, missing letters, e.t.c
"""
# df['Status'].unique()
# df['Vehicle_Type'].unique()
# df['Priority'].unique()
# df['Origin_City'].unique()
# df['Destination'].unique()
# df['Insurance'].unique()
# df['Payment_Method'].unique()

category_columns = ['Status', 'Vehicle_Type', 'Priority', 'Origin_City', 'Destination', 'Insurance', 'Payment_Method']
for column in category_columns:
    print(f'\n{column}: {df[column].unique()}')


Status: ['Delivered' 'In Transit' 'IN TRANSIT' 'Cancelled' 'delivered ' 'Delayed'
 'Delayd']

Vehicle_Type: ['Van' 'Bke' 'VAN ' 'Truck' 'truck' 'Bike']

Priority: ['High' 'Medium' 'high ' 'Low' 'LOW']

Origin_City: ['Abuja' 'Lagos' 'Port Harcourt' 'ABUJA' 'lagos ' 'Kano']

Destination: ['Customer Hub' 'Retail Store' 'customer hub']

Insurance: ['No' 'Yes' 'NO' 'yes ']

Payment_Method: ['Cash' 'CARD' 'cash ' 'Card' 'Transfer' 'Tranfer']


In [8]:
""" checking the rows with negative values in Cost_NGN, Delivery_Days and Weight_kg """
""" The negative values are error. """

negative_Weight = df['Weight_kg'].value_counts()
negative_Weight = negative_Weight[negative_Weight.index < 0]
negative_Weight.index

negative_Cost = df['Cost_NGN'].value_counts()
negative_Cost = negative_Cost[negative_Cost.index < 0]
negative_Cost.index

negative_Days = df['Delivery_Days'].value_counts()
negative_Days = negative_Days[negative_Days.index < 0]
negative_Days.index

print('Negative Weight_kg rows:', (df['Weight_kg'] < 0).sum())
print('Negative Cost_NGN rows:', (df['Cost_NGN'] < 0).sum())
print('Negative Delivery_Days rows:', (df['Delivery_Days'] < 0).sum())
print('Negative Distance_km rows:', (df['Distance_km'] < 0).sum())


Negative Weight_kg rows: 4995
Negative Cost_NGN rows: 997
Negative Delivery_Days rows: 2942
Negative Distance_km rows: 3425


In [9]:
""" The Notes column seems to have little or no real impact. hence we should remove it"""
df['Notes'].unique()

array(['On time delivery', '  note '], dtype=object)

In [10]:
""" For the Remarks column; how many nulls does it have, and what are the unique values? 
i'd suggest we drop it
 """
# df['Remarks'].isnull().sum()
# df['Remarks'].unique()
print(f'The column has {df['Remarks'].isnull().sum()} null values and its unique values are: {df['Remarks'].unique()}')

The column has 37337 null values and its unique values are: ['Delayed delivery' 'Good condition' nan 'Damaged']


In [11]:
""" I notice the negative value of 'min' in 'Weight_kg', 'Cost_NGN', 'Distance_km' and 'Delivery_Days' """

df.describe().loc[['min', 'max', 'mean', 'std'], ['Weight_kg', 'Cost_NGN', 'Distance_km', 'Delivery_Days', 'Customer_Rating']]

,Weight_kg,Cost_NGN,Distance_km,Delivery_Days,Customer_Rating
min,-25.740000,-61892.500000,-362.080000,-10.000000,1.000000
max,32.400000,590875.500000,984.320000,10.000000,5.000000
mean,9.800561,7472.966621,299.965635,5.298791,3.005455
std,5.404105,26647.598872,149.773822,3.247309,1.414116
